# Custom SKU Environment Setup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/dreamprice/blob/master/notebooks/custom_env.ipynb)

This notebook demonstrates how to configure the `GroceryPricingEnv` with
custom SKU cost vectors, run manual pricing strategies, and compare
policies. Uses the built-in log-linear demand fallback (`world_model=None`).

> **Note:** For production-grade evaluation, load the trained DreamPrice
> checkpoint from [HuggingFace](https://huggingface.co/qbz506/dreamprice-cso)
> and pass it as `world_model=loaded_model`.

In [ ]:
# Install DreamPrice (skip if already installed)
import subprocess, sys
try:
    import retail_world_model
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '..'])

In [ ]:
import numpy as np
from retail_world_model.envs.grocery import GroceryPricingEnv, PRICE_MULTIPLIERS

# Define 5 SKUs with custom cost vectors
cost_vector = np.array([1.80, 0.90, 2.10, 0.60, 3.00])

env = GroceryPricingEnv(
    world_model=None,
    store_features=np.zeros(8),
    initial_obs=np.zeros(32),
    cost_vector=cost_vector,
    n_skus=5,
    H=13,
)
print(f'Environment created with {env.n_skus} SKUs')
print(f'Cost vector: {cost_vector}')
print(f'Action space: {env.action_space}')

## Action Space

The action space is `MultiDiscrete([21] * n_skus)` — each SKU gets an
independent discrete action selecting a price multiplier:

| Action | Multiplier | Meaning |
|--------|-----------|----------|
| 0      | 0.90      | -10% discount |
| 5      | 0.95      | -5% discount |
| 10     | 1.00      | No change |
| 15     | 1.05      | +5% markup |
| 20     | 1.10      | +10% markup |

In [ ]:
# Step through the environment manually
obs, info = env.reset(seed=0)

# Keep prices flat for 5 weeks, then discount SKU 0
actions = [
    np.array([10, 10, 10, 10, 10]),  # week 1: no change
    np.array([10, 10, 10, 10, 10]),  # week 2: no change
    np.array([10, 10, 10, 10, 10]),  # week 3: no change
    np.array([10, 10, 10, 10, 10]),  # week 4: no change
    np.array([10, 10, 10, 10, 10]),  # week 5: no change
    np.array([ 5, 10, 10, 10, 10]),  # week 6: SKU 0 at -5%
    np.array([ 3, 10, 10, 10, 10]),  # week 7: SKU 0 at -7%
]

for i, action in enumerate(actions):
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'Week {i+1}: reward={reward:.3f}, prices={info["prices"]}')

In [ ]:
# Policy comparison: random vs hold vs aggressive-discount
import matplotlib.pyplot as plt

def run_episodes(env, policy_fn, n_episodes=20):
    returns = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        total_reward = 0.0
        done = False
        while not done:
            action = policy_fn(env)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        returns.append(total_reward)
    return np.array(returns)

random_ret = run_episodes(env, lambda e: e.action_space.sample())
hold_ret = run_episodes(env, lambda e: np.full(e.n_skus, 10, dtype=int))
discount_ret = run_episodes(env, lambda e: np.full(e.n_skus, 3, dtype=int))

print(f'Random:   mean={random_ret.mean():.2f} +/- {random_ret.std():.2f}')
print(f'Hold:     mean={hold_ret.mean():.2f} +/- {hold_ret.std():.2f}')
print(f'Discount: mean={discount_ret.mean():.2f} +/- {discount_ret.std():.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
positions = [1, 2, 3]
data = [random_ret, hold_ret, discount_ret]
bp = ax.boxplot(data, positions=positions, widths=0.5, patch_artist=True)
colors = ['#4C72B0', '#55A868', '#C44E52']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xticks(positions)
ax.set_xticklabels(['Random', 'Hold', 'Deep Discount'])
ax.set_ylabel('Episode Return')
ax.set_title('Policy Comparison (20 episodes each)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()